[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C24_Inference_Serving_Course/04_prefix_disagg/04_prefix_disagg.ipynb)

# 04 · Prefix Cache 与 PD 分离（用 numpy 做模拟器+账本）

目标：把 **前缀复用**、**RadixAttention 的 radix 前缀树**、**带引用计数的 LRU 淘汰**、**PD 分离的资源配比与 TTFT/TPOT 分解** 用 numpy/python 实现出来，并用 `assert` 钉死核心不变量。

路线：暴力前缀匹配（参考）→ radix 前缀树（match_prefix **对拍**暴力 lcp）→ 前缀命中率与省下的 prefill → 带引用计数的 LRU 淘汰 → PD 资源配比账 → TTFT/TPOT 分解（colocate vs disaggregate）→ ✏️ 练习 → 📖 答案 → 🧪 真实负载胶囊。

> 心智模型：**radix 树 = 自动发现公共前缀的字典；命中即跳过 prefill；PD 分离 = 把算力受限的 prefill 和带宽受限的 decode 拆开各自达标**。核心不变量：**树的 match_prefix 逐字等于暴力最长前缀匹配**。

## 1 · 暴力前缀匹配（ground truth）

前缀缓存要回答：一个新请求的 prompt，有多长一段前缀已经在缓存里？
先写**绝对可信的暴力版**：对缓存里每个已存序列，逐位比较公共前缀长度，取最大值。这是后面 radix 树要对拍的 ground truth。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def lcp_len(a, b):
    '''两个 token 序列的最长公共前缀长度。'''
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n

def brute_match(cache_seqs, query):
    '''query 在缓存里能命中的最长前缀长度 = 对每个已存序列取 lcp 的最大值。'''
    return max((lcp_len(query, s) for s in cache_seqs), default=0)

# token 用整数表示（模拟 token id）
cache = [
    [1, 2, 3, 4, 5],          # system + 任务A
    [1, 2, 3, 9, 9],          # 同 system，任务不同
    [7, 7, 7],                # 完全不同
]
q = [1, 2, 3, 4, 8]            # 与第1条共享前 4 个 token
m = brute_match(cache, q)
print(f'query={q}  最长命中前缀长度 = {m}')
assert m == 4, '应命中 [1,2,3,4]'
assert brute_match(cache, [9, 9]) == 0, '无公共前缀 -> 0'
assert brute_match([], [1, 2]) == 0, '空缓存 -> 0'
print('✅ 暴力前缀匹配就绪（ground truth）')

## 2 · radix 前缀树：match_prefix 对拍暴力 lcp（核心不变量）

把所有缓存序列组织成一棵 **radix 树**：每条边携带一段 token，根到节点的路径 = 一个被缓存前缀。
实现 `insert(tokens)`（含**边裂开 split**）与 `match_prefix(tokens)`（沿树走最长前缀）。

**核心不变量**：对任意 query，`tree.match_prefix(q)` 必须**逐字等于** `brute_match(cache, q)`。

In [ ]:
class RadixNode:
    def __init__(self):
        self.children = {}   # 首 token -> (edge_tokens(list), child RadixNode)

class RadixTree:
    def __init__(self):
        self.root = RadixNode()

    def insert(self, tokens):
        node, i = self.root, 0
        tokens = list(tokens)
        while i < len(tokens):
            first = tokens[i]
            if first not in node.children:
                # 没有以该 token 开头的边 -> 新建一条携带剩余后缀的边
                leaf = RadixNode()
                node.children[first] = (tokens[i:], leaf)
                return
            edge, child = node.children[first]
            # 在这条边内逐 token 比对
            j = 0
            while j < len(edge) and i < len(tokens) and edge[j] == tokens[i]:
                j += 1; i += 1
            if j == len(edge):
                node = child                 # 整条边匹配，进入下一节点
                continue
            # 边只匹配了前 j 个 token -> 在 j 处把边裂成两段
            mid = RadixNode()
            mid.children[edge[j]] = (edge[j:], child)   # 旧边的后半挂到 mid 下
            node.children[first] = (edge[:j], mid)      # 前半连到 mid
            if i < len(tokens):                          # query 剩余后缀从 mid 分叉
                leaf = RadixNode()
                mid.children[tokens[i]] = (tokens[i:], leaf)
            return

    def match_prefix(self, tokens):
        node, i = self.root, 0
        tokens = list(tokens)
        while i < len(tokens):
            first = tokens[i]
            if first not in node.children:
                break
            edge, child = node.children[first]
            j = 0
            while j < len(edge) and i < len(tokens) and edge[j] == tokens[i]:
                j += 1; i += 1
            if j < len(edge):
                break                        # 边只匹配一部分 -> 停在边中间
            node = child
        return i

tree = RadixTree()
for s in cache:
    tree.insert(s)
print('match_prefix([1,2,3,4,8]) =', tree.match_prefix([1, 2, 3, 4, 8]))
assert tree.match_prefix([1, 2, 3, 4, 8]) == brute_match(cache, [1, 2, 3, 4, 8])
assert tree.match_prefix([1, 2, 3, 9, 0]) == brute_match(cache, [1, 2, 3, 9, 0])
assert tree.match_prefix([7, 7, 7, 7]) == brute_match(cache, [7, 7, 7, 7])
print('✅ 单点对拍通过（含边裂开的情形）')

现在用**随机化测试**狠狠对拍：随机建缓存、随机查，断言树版与暴力版对**每一个** query 都逐字相等。

In [ ]:
def random_seq(rng, maxlen=8, vocab=4):
    n = int(rng.integers(1, maxlen + 1))
    return [int(x) for x in rng.integers(0, vocab, size=n)]

fails = 0
for trial in range(300):
    rg = np.random.default_rng(trial)
    cache_seqs = [random_seq(rg) for _ in range(int(rg.integers(1, 8)))]
    t = RadixTree()
    for s in cache_seqs:
        t.insert(s)
    for _ in range(10):
        q = random_seq(rg)
        if t.match_prefix(q) != brute_match(cache_seqs, q):
            fails += 1
assert fails == 0, f'树版与暴力版不一致 {fails} 次'
print('✅ 300 棵随机树 × 10 query：radix match_prefix 逐字等于暴力 lcp —— 数据结构正确性铁证')

## 3 · 前缀命中率与省下的 prefill

缓存的价值 = 命中的 token 不用再 prefill。喂一批带**共享前缀**的请求（典型：同一 system prompt），
边查边插：每个请求先 `match_prefix` 看命中多少，命中部分跳过 prefill，未命中部分才 prefill 并插入缓存。
断言：带共享前缀时命中率 > 0，且**省下的 prefill 等于命中长度之和**。

In [ ]:
def serve_with_cache(requests):
    '''按到达顺序处理请求：返回 (总命中token, 总prompttoken, 总实际prefill).'''
    tree = RadixTree()
    hit_total = prompt_total = prefill_total = 0
    for q in requests:
        hit = tree.match_prefix(q)        # 命中的前缀长度
        new = len(q) - hit                # 未命中、需要 prefill 的后缀
        hit_total += hit
        prompt_total += len(q)
        prefill_total += new
        tree.insert(q)                    # 把整条 query 存入缓存供后续复用
    return hit_total, prompt_total, prefill_total

SYS = [1, 1, 1, 1, 1, 1]                  # 6-token 公共 system prompt
requests = [SYS + [i, i] for i in range(2, 8)]   # 同前缀，各带不同 2-token 后缀
hit, prompt, prefill = serve_with_cache(requests)
rate = hit / prompt
print(f'命中 {hit}/{prompt} token  命中率={rate:.0%}  实际 prefill={prefill} token')
# 不用缓存就要 prefill 全部 prompt；省下的 = 命中数
saved = prompt - prefill
assert saved == hit, '省下的 prefill 必须等于命中的 token 数'
assert rate > 0.5, '共享长前缀 -> 命中率应很高'
print(f'✅ 缓存省下 {saved} token 的 prefill（= 命中数）；共享前缀越长省得越多 -> TTFT 越低')

再看**多轮对话**：第 k 轮的前缀 = 前 k−1 轮全部，命中率随轮数趋近 1。

In [ ]:
def multiturn(n_turns, turn_len=3):
    '''构造多轮对话：第 k 轮 prompt = 前面所有轮拼接 + 本轮新增。'''
    reqs, hist = [], []
    for k in range(n_turns):
        new = [10 + k] * turn_len          # 本轮新增 token
        reqs.append(hist + new)            # 本轮 prompt = 历史 + 新增
        hist = hist + new                 # 更新历史
    return reqs

reqs = multiturn(5, turn_len=3)
hit, prompt, prefill = serve_with_cache(reqs)
print(f'多轮: 命中率={hit/prompt:.0%}, 实际 prefill={prefill} (每轮只 prefill 新增的 3 token)')
assert prefill == 5 * 3, '每轮只需 prefill 本轮新增的 token'
print('✅ 多轮对话：除最新一轮外几乎全部命中 —— 这是前缀缓存最甜的场景')

## 4 · 带引用计数的 LRU 淘汰

缓存的 KV 也占显存。缓存满时要淘汰，但**正在被某请求使用的前缀不能淘汰**（refcount>0），
只能从 **refcount==0** 的条目里挑**最久未用（LRU）**的删。我们用一个简化的「前缀块缓存」演示这条规则。

In [ ]:
class RefCountedLRU:
    def __init__(self, capacity):
        self.capacity = capacity
        self.refcount = {}      # key -> 正在用它的请求数
        self.last_used = {}     # key -> 逻辑时间戳
        self.clock = 0

    def _evict_if_needed(self):
        while len(self.refcount) > self.capacity:
            # 只能淘汰 refcount==0 的；在它们里挑 last_used 最小(最久未用)
            free_keys = [k for k, rc in self.refcount.items() if rc == 0]
            assert free_keys, 'OOM: 所有缓存项都在使用中，无法淘汰'
            victim = min(free_keys, key=lambda k: self.last_used[k])
            del self.refcount[victim]; del self.last_used[victim]

    def acquire(self, key):
        '''请求开始使用某前缀：refcount+1，更新时间戳，必要时淘汰别人。'''
        self.clock += 1
        self.refcount[key] = self.refcount.get(key, 0) + 1
        self.last_used[key] = self.clock
        self._evict_if_needed()

    def release(self, key):
        '''请求用完某前缀：refcount-1（可被淘汰，但暂不删）。'''
        self.refcount[key] -= 1

    def __contains__(self, key):
        return key in self.refcount

lru = RefCountedLRU(capacity=2)
lru.acquire('A'); lru.release('A')      # A 用完，可淘汰
lru.acquire('B')                        # B 正在用 (refcount=1)
lru.acquire('C')                        # 容量2，插入C -> 淘汰 refcount==0 里最久的 = A
assert 'A' not in lru and 'B' in lru and 'C' in lru
print('插入 C 触发淘汰：A(空闲最久) 被踢, B(在用) 保留 ->', sorted(lru.refcount))
print('✅ LRU 淘汰只动 refcount==0 的最久未用项，正在用的前缀安全')

关键安全性：**正在使用的前缀绝不被淘汰**，哪怕它最久未访问。

In [ ]:
lru2 = RefCountedLRU(capacity=2)
lru2.acquire('X')                       # X 一直被持有 (refcount=1)，且会变成最久未用
lru2.acquire('Y'); lru2.release('Y')
lru2.acquire('Z')                       # 满了要淘汰：X 虽最久但在用 -> 只能踢 Y
assert 'X' in lru2, '正在使用的前缀(refcount>0)绝不能被淘汰'
assert 'Y' not in lru2
print('✅ X 虽是最久未用，但 refcount>0 受保护；被踢的是空闲的 Y —— 这正是 RadixAttention 的淘汰纪律')

## 5 · PD 分离的资源配比账

PD 分离把 prefill 和 decode 拆到两个池。配比原则：让两池**吞吐相匹配**，谁都不堆积。
一个请求在 prefill 上花的时间 ∝ `prompt_len / r_prefill`，在 decode 上花 ∝ `gen_len / r_decode`；
两池实例数之比 = 这两个时间之比。

In [ ]:
def pd_ratio(prompt_len, gen_len, r_prefill, r_decode):
    '''返回 (每请求 prefill 耗时, 每请求 decode 耗时, 平衡所需的 N_prefill : N_decode).'''
    t_prefill = prompt_len / r_prefill
    t_decode  = gen_len   / r_decode
    return t_prefill, t_decode, t_prefill / t_decode

# prefill 一次处理整段(算力受限但批量大) -> 速率高; decode 逐token(带宽受限) -> 速率低
tp, td, ratio = pd_ratio(prompt_len=1000, gen_len=200, r_prefill=5000.0, r_decode=100.0)
print(f'每请求 prefill 耗时={tp:.3f}s, decode 耗时={td:.3f}s')
print(f'平衡配比 N_prefill : N_decode = {ratio:.2f} : 1')
# tp=1000/5000=0.2, td=200/100=2.0 -> 比值=0.1 -> decode 池要 10 倍于 prefill
assert abs(tp - 0.2) < 1e-9 and abs(td - 2.0) < 1e-9
assert abs(ratio - 0.1) < 1e-9, 'decode 慢得多 -> 需要远多于 prefill 的 decode 实例'
print('✅ decode 是慢工序 -> decode 池需 10x 实例。配比就是让两工序产能对齐。')

In [ ]:
# 把比值换算成给定总 GPU 数下的整数配比
def allocate_pd(total_gpus, prompt_len, gen_len, r_prefill, r_decode):
    _, _, ratio = pd_ratio(prompt_len, gen_len, r_prefill, r_decode)
    # N_p / N_d = ratio, N_p + N_d = total -> N_p = total*ratio/(1+ratio)
    n_p = max(1, round(total_gpus * ratio / (1 + ratio)))
    n_d = total_gpus - n_p
    return n_p, n_d

n_p, n_d = allocate_pd(11, 1000, 200, 5000.0, 100.0)
print(f'11 张卡 -> prefill {n_p} 张, decode {n_d} 张')
assert n_p + n_d == 11 and n_d > n_p, 'decode 该分到更多卡'
print('✅ 按平衡式把总卡数切成 prefill:decode 两池')

## 6 · TTFT/TPOT 分解：colocate vs disaggregate

**TTFT 由 prefill 决定，TPOT 由 decode 决定**。混跑（colocate）时，一次长 prefill 会插队、卡住正在 decode 的请求，
推高它们的 TPOT（抖动）；分离（disaggregate）后 decode 池不被 prefill 打断，TPOT 更稳。我们用一个最小模型量化这点。

In [ ]:
def tpot_samples(mode, n_decode_steps=20, base_tpot=0.05,
                 prefill_burst=0.5, prefill_every=5):
    '''模拟 decode 过程中每步的 TPOT。
       colocate: 每隔 prefill_every 步，有一次长 prefill 插队，这一步 TPOT 暴涨。
       disaggregate: prefill 在别的池，decode 步步平稳。'''
    out = []
    for step in range(n_decode_steps):
        t = base_tpot
        if mode == 'colocate' and step % prefill_every == 0:
            t += prefill_burst          # 被一次 prefill 插队拖慢
        out.append(t)
    return np.array(out)

co = tpot_samples('colocate')
di = tpot_samples('disaggregate')
print(f"colocate     TPOT: 均值={co.mean():.3f}s  P-max={co.max():.3f}s  抖动(std)={co.std():.3f}")
print(f"disaggregate TPOT: 均值={di.mean():.3f}s  P-max={di.max():.3f}s  抖动(std)={di.std():.3f}")
assert di.std() < co.std(), '分离后 TPOT 抖动应更小'
assert di.max() < co.max(), '分离后没有 prefill 插队的尖峰'
print('✅ 分离消除了 prefill 对 decode 的插队干扰 -> TPOT 更稳、尾延迟更低（goodput 更高）')

---
## ✏️ 练习 1：最长前缀匹配（暴力版）

实现 `longest_prefix(cache_seqs, query)`：返回 `query` 在 `cache_seqs` 里命中的最长前缀长度。
（不准用上面的 `brute_match`/`lcp_len`，自己写。）空缓存返回 0。

In [ ]:
def longest_prefix(cache_seqs, query):
    # TODO: 对每个已存序列，逐位比 query 的公共前缀长度，取最大；空缓存->0
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
cache = [[1, 2, 3, 4], [1, 2, 9], [5, 6]]
assert longest_prefix(cache, [1, 2, 3, 7]) == 3      # 命中 [1,2,3]
assert longest_prefix(cache, [1, 2, 9, 9]) == 3      # 命中 [1,2,9]
assert longest_prefix(cache, [9, 9]) == 0
assert longest_prefix([], [1]) == 0
assert longest_prefix(cache, [5, 6, 7]) == 2
print('✅ 练习 1 通过：最长前缀匹配正确')

## ✏️ 练习 2：前缀命中率

实现 `hit_rate(requests)`：按到达顺序处理请求（用第 2 节的 `RadixTree`），
每个请求先 `match_prefix` 记命中、再 `insert`，返回 `(总命中token, 总prompttoken, 命中率)`。

In [ ]:
def hit_rate(requests):
    # TODO: 建一棵 RadixTree，按序对每个 req 先 match_prefix(累加命中) 再 insert
    #       返回 (hit_total, prompt_total, hit_total/prompt_total)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
SYS = [1, 1, 1, 1]
reqs = [SYS + [k] for k in range(2, 7)]               # 共享 4-token 前缀
ht, pt, rate = hit_rate(reqs)
# 第1个请求全 miss(0命中)，之后每个命中 4 -> 命中 = 4*(len-1)
assert ht == 4 * (len(reqs) - 1), '除首个外每个命中 4 token'
assert pt == sum(len(r) for r in reqs)
assert abs(rate - ht / pt) < 1e-12 and rate > 0
print(f'✅ 练习 2 通过：命中率={rate:.0%}（{ht}/{pt}）')

## ✏️ 练习 3：PD 配比闭式

实现 `decode_heavy_ratio(prompt_len, gen_len, r_prefill, r_decode)`：返回 `N_decode / N_prefill`
（即每 1 个 prefill 实例需要配几个 decode 实例）。这是第 5 节比值的倒数。

In [ ]:
def decode_heavy_ratio(prompt_len, gen_len, r_prefill, r_decode):
    # TODO: t_prefill = prompt_len/r_prefill; t_decode = gen_len/r_decode
    #       返回 t_decode / t_prefill  (= N_decode / N_prefill)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
r = decode_heavy_ratio(prompt_len=1000, gen_len=200, r_prefill=5000.0, r_decode=100.0)
# t_prefill=0.2, t_decode=2.0 -> 2.0/0.2 = 10
assert abs(r - 10.0) < 1e-9, '每个 prefill 实例配 10 个 decode 实例'
# 生成越长，decode 越吃紧 -> 需要更多 decode 实例
r_long = decode_heavy_ratio(1000, 2000, 5000.0, 100.0)
assert r_long > r, '生成更长 -> decode 池需更大'
print(f'✅ 练习 3 通过：每 prefill 实例配 {r:.0f} 个 decode 实例；生成越长 decode 池越大')

## ✏️ 练习 4：TTFT 随命中率下降

实现 `ttft(prompt_len, hit_len, prefill_rate)`：TTFT ≈ 需要 prefill 的 token 数 ÷ prefill 速率。
命中的前缀跳过 prefill，所以只 prefill `prompt_len - hit_len` 个 token。

In [ ]:
def ttft(prompt_len, hit_len, prefill_rate):
    # TODO: 只对未命中的 (prompt_len - hit_len) 个 token 做 prefill
    #       返回 (prompt_len - hit_len) / prefill_rate
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
no_cache = ttft(prompt_len=2000, hit_len=0,    prefill_rate=5000.0)
cached   = ttft(prompt_len=2000, hit_len=1900, prefill_rate=5000.0)
print(f'无缓存 TTFT={no_cache*1000:.0f}ms, 命中1900 后 TTFT={cached*1000:.0f}ms')
assert abs(no_cache - 0.4) < 1e-9        # 2000/5000
assert abs(cached - 0.02) < 1e-9         # 100/5000
assert cached < no_cache / 10, '高命中率应把 TTFT 砍一个数量级'
print('✅ 练习 4 通过：命中率越高，TTFT 越低（这里从 400ms 降到 20ms）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def longest_prefix(cache_seqs, query):
    best = 0
    for s in cache_seqs:
        n = 0
        for x, y in zip(query, s):
            if x != y:
                break
            n += 1
        best = max(best, n)
    return best

In [ ]:
# 练习 2 参考答案
def hit_rate(requests):
    tree = RadixTree()
    hit_total = prompt_total = 0
    for q in requests:
        hit_total += tree.match_prefix(q)
        prompt_total += len(q)
        tree.insert(q)
    return hit_total, prompt_total, hit_total / prompt_total

In [ ]:
# 练习 3 参考答案
def decode_heavy_ratio(prompt_len, gen_len, r_prefill, r_decode):
    t_prefill = prompt_len / r_prefill
    t_decode = gen_len / r_decode
    return t_decode / t_prefill

In [ ]:
# 练习 4 参考答案
def ttft(prompt_len, hit_len, prefill_rate):
    return (prompt_len - hit_len) / prefill_rate

---
## 🧪 真实数据胶囊：真实负载下前缀复用省多少

用**真实量级**的服务负载算一笔账：长公共 system prompt + 短用户问题，是 LLM 服务最常见的形态。
真实系统（如 SGLang RadixAttention、DistServe）在这种负载下报告了显著的 TTFT 下降与 goodput 提升。

> 这里用代表性的真实数字（system prompt 数百~上千 token、用户问题数十 token）；联网不可用时回退到内置数值。

In [ ]:
# 代表性真实负载（token 数为典型量级）
def load_real_workload():
    try:
        # 真实环境可在此从 tokenizer 统计真实 system prompt 长度等
        raise RuntimeError('offline')
    except Exception:
        # 内置真实量级回退：典型 RAG/agent 服务
        return dict(system_prompt=800, few_shot=1200, user_question=40, n_requests=100)

w = load_real_workload()
shared_prefix = w['system_prompt'] + w['few_shot']     # 所有请求共享
per_req = shared_prefix + w['user_question']
print(f"共享前缀={shared_prefix} token, 每请求总长={per_req} token, 请求数={w['n_requests']}")

# 无缓存：每个请求都 prefill 全部；有缓存：仅首个 prefill 前缀，其余只 prefill 各自问题
no_cache = per_req * w['n_requests']
with_cache = per_req + (per_req - shared_prefix) * (w['n_requests'] - 1)  # 首个全算+其余只算问题
print(f'无缓存总 prefill = {no_cache} token')
print(f'有缓存总 prefill = {with_cache} token  (省 {100*(1-with_cache/no_cache):.0f}%)')
assert with_cache < no_cache * 0.1, '长共享前缀 -> 缓存省掉 >90% 的 prefill'
print('✅ 长 system prompt + 短问题：前缀复用省掉绝大部分 prefill 算力 -> TTFT 大降')

**🧪 胶囊练习**：实现 `prefill_saving(shared_prefix, question_len, n_requests)`：
返回 `(无缓存总prefill, 有缓存总prefill, 省下比例)`。规则：无缓存每个请求 prefill `shared_prefix+question_len`；
有缓存只有第 1 个请求 prefill 前缀，其余只 prefill 各自 `question_len`。

In [ ]:
def prefill_saving(shared_prefix, question_len, n_requests):
    # TODO: no_cache = (shared_prefix+question_len)*n_requests
    #       with_cache = (shared_prefix+question_len) + question_len*(n_requests-1)
    #       返回 (no_cache, with_cache, 1 - with_cache/no_cache)
    raise NotImplementedError

In [ ]:
# 自测
no_c, with_c, saved = prefill_saving(shared_prefix=800, question_len=40, n_requests=100)
assert no_c == 840 * 100
assert with_c == 840 + 40 * 99
assert saved > 0.9, '长共享前缀应省下 >90%'
print(f'无缓存={no_c}, 有缓存={with_c}, 省下 {saved:.0%}')
print('✅ 胶囊练习通过：共享前缀越长、请求越多，前缀复用省得越多')

In [ ]:
# 📖 胶囊参考答案
def prefill_saving(shared_prefix, question_len, n_requests):
    per_req = shared_prefix + question_len
    no_cache = per_req * n_requests
    with_cache = per_req + question_len * (n_requests - 1)
    return no_cache, with_cache, 1 - with_cache / no_cache

---
### 小结
- 共享前缀（system prompt / few-shot / 多轮历史）的 KV 是**确定性、可缓存**的；缓存它就跳过重复 prefill，直接砍 **TTFT**。
- **RadixAttention** = radix 前缀树：公共前缀 = 共享路径，自动、跨请求、细粒度复用；带引用计数的 LRU 淘汰保护在用前缀。
- 核心不变量：树的 **match_prefix 逐字等于暴力最长前缀匹配**（300 棵随机树验证）。
- **PD 分离**把算力受限的 prefill 与带宽受限的 decode 拆到两个池，消除阶段干扰；配比 = 两阶段耗时之比。
- **SLO 分解**：TTFT↔prefill（前缀复用/PD），TPOT↔decode（大 batch/投机/量化/隔离）。

下一站：**模块 05 · fp8 与量化服务** —— decode 的最后一块带宽，靠把 KV 和权重降到 8 bit 来省。